####Requirement - Analysis Data Set

Prepare club bookings dataset for analysis
```
+----------+------------+---------------+-------------------+--------------+
|booking_id| member_name|  facility_name|         start_time|booking_amount|
+----------+------------+---------------+-------------------+--------------+
```

In [0]:
# Aggregation is the final step in DE because we require the well prepared or transformed data to answer the business questions.
bookings_df = spark.table("dev.spark_db.bookings")
facilities_df = spark.table("dev.spark_db.facilities")
members_df = spark.table("dev.spark_db.members")

club_bookings_df = (
    bookings_df.join(facilities_df, "facility_id")
            .join(members_df, "member_id", "left")
            .selectExpr("booking_id",
                        "case when member_id==0 then 'Guest Member' else concat_ws(' ', first_name, last_name) end as member_name",
                        "facility_name","start_time",
                        "case when member_id == 0 then slots * guest_cost else slots * member_cost end as booking_amount")
)

club_bookings_df.display()

Q1. Who are the top 5 members by total booking amount?

Prepare a report as the following.
```
member_name     | total_booking_amount
---------------------------------------
Tim Rownam      | 6480
Tim Boothe      | 3644
Gerald Butters  | 3343
Burton Tracy    | 2953
David Jones     | 2651
```

1.1 Try aggregation using select or selectExpr

In [0]:

result_df =(
    club_bookings_df.where("member_name != 'Guest Member'")
                .groupBy("member_name")
                .selectExpr("member_name", "sum(booking_amount) as total_booking_amount")
)

result_df.display()

1.2 Try using agg() transformation

In [0]:
from pyspark.sql.functions import expr,col
result_df =(
    club_bookings_df.where("member_name != 'Guest Member'")
                .groupBy("member_name")
                .agg(expr("sum(booking_amount) as total_booking_amount"))
                #.orderBy("total_booking_amount", ascending=False)
                .orderBy(col("total_booking_amount").desc_nulls_last())
                .limit(5)
)

result_df.display()

Q2. Who are the members having total booking amount > 2500?

In [0]:
from pyspark.sql.functions import expr,col
result_df =(
    club_bookings_df.where("member_name != 'Guest Member'")
                .groupBy("member_name")
                .agg(expr("sum(booking_amount) as total_booking_amount"))
                .orderBy(col("total_booking_amount").desc_nulls_last())
                .where("total_booking_amount > 2500") # In Sql its having clause for aggregation but in pyspark we can use where as well. where should come after aggregations to make it work.
)
result_df.display()

Q3. Find member wise facility bookings for more than 2500?
```
+-----------+--------------+--------------------+
|member_name| facility_name|total_booking_amount|
+-----------+--------------+--------------------+
| Tim Boothe|Massage Room 1|              2660.0|
| Tim Rownam|Massage Room 1|              6160.0|
+-----------+--------------+--------------------+
```


In [0]:
from pyspark.sql.functions import expr, col

result_df = (
    club_bookings_df.where("member_name != 'Guest Member'")
            .groupBy("member_name", "facility_name")
            .agg(expr("sum(booking_amount) as total_booking_amount"))
            .where("total_booking_amount > 2500")
)

result_df.display()